# Homework 3 - NYC taxi demand dataset

In [1]:
import polars as pl

DATA_DIR = "data"

MONEY_COLS = [
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "airport_fee",
]
VENDOR_IDS = [1, 2, 6, 7]
RATECODE_IDS = [1, 2, 3, 4, 5, 6, 99]

BOROUGH = pl.Enum(
    [
        "bronx",
        "brooklyn",
        "ewr",
        "manhattan",
        "queens",
        "staten_island",
        "unknown",
        "outside_nyc",
    ]
)
PAYMENT_TYPE = pl.Enum(["card", "cash", "other"])

## 1. Data reading

In [2]:
def read_rides() -> pl.LazyFrame:
    months = [
        pl.scan_parquet(f"{DATA_DIR}/{month}.parquet").with_columns(
            pl.col("tpep_pickup_datetime", "tpep_dropoff_datetime").dt.cast_time_unit(
                "us"
            )
        )
        for month in range(1, 13)
    ]
    return (
        pl.concat(months)
        .rename({"Airport_fee": "airport_fee"})
        .filter(
            pl.col("tpep_pickup_datetime").dt.year() == 2024,
            pl.col("tpep_dropoff_datetime") <= pl.datetime(2025, 1, 1),
        )
        .with_columns(
            pl.col("VendorID", "RatecodeID", "passenger_count", "payment_type").cast(
                pl.UInt8
            ),
            pl.col("PULocationID", "DOLocationID").cast(pl.UInt16),
            pl.col("store_and_fwd_flag").cast(pl.Enum(["N", "Y"])),
        )
    )


def read_zones() -> pl.LazyFrame:
    return pl.scan_csv(f"{DATA_DIR}/taxi_zone_lookup.csv").select(
        pl.col("LocationID").cast(pl.UInt16),
        pl.col("Borough")
        .replace("N/A", "outside_nyc")
        .str.to_lowercase()
        .str.replace(" ", "_")
        .cast(BOROUGH)
        .alias("borough"),
    )

In [3]:
read_rides().collect_schema()

Schema([('VendorID', UInt8),
        ('tpep_pickup_datetime', Datetime(time_unit='us', time_zone=None)),
        ('tpep_dropoff_datetime', Datetime(time_unit='us', time_zone=None)),
        ('passenger_count', UInt8),
        ('trip_distance', Float64),
        ('RatecodeID', UInt8),
        ('store_and_fwd_flag', Enum(categories=['N', 'Y'])),
        ('PULocationID', UInt16),
        ('DOLocationID', UInt16),
        ('payment_type', UInt8),
        ('fare_amount', Float64),
        ('extra', Float64),
        ('mta_tax', Float64),
        ('tip_amount', Float64),
        ('tolls_amount', Float64),
        ('improvement_surcharge', Float64),
        ('total_amount', Float64),
        ('congestion_surcharge', Float64),
        ('airport_fee', Float64)])

In [4]:
read_zones().collect()

LocationID,borough
u16,enum
1,"""ewr"""
2,"""queens"""
3,"""bronx"""
4,"""manhattan"""
5,"""staten_island"""
…,…
261,"""manhattan"""
262,"""manhattan"""
263,"""manhattan"""


## 2. Data cleaning and filtering

In [5]:
def clean(rides: pl.LazyFrame) -> pl.LazyFrame:
    trip_duration = pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime")
    return rides.with_columns(
        pl.col("passenger_count").fill_null(1).clip(upper_bound=6),
        pl.col(MONEY_COLS).fill_null(0).abs(),
    ).filter(
        pl.col("passenger_count") > 0,
        trip_duration <= pl.duration(hours=2),
        ~pl.any_horizontal(pl.col(MONEY_COLS) > 1000),
        pl.col("VendorID").is_in(VENDOR_IDS),
        pl.col("RatecodeID").is_in(RATECODE_IDS),
    )

In [6]:
rows_raw = read_rides().select(pl.len()).collect().item()
rows_clean = read_rides().pipe(clean).select(pl.len()).collect().item()

print(f"Rows before cleaning: {rows_raw}")
print(f"Rows after cleaning: {rows_clean} ({rows_clean / rows_raw:.2%})")

Rows before cleaning: 41169064
Rows after cleaning: 36635240 (88.99%)


## 3. Data transformation

In [7]:
def transform(rides: pl.LazyFrame, zones: pl.LazyFrame) -> pl.LazyFrame:
    pickup_time = pl.col("tpep_pickup_datetime").dt.time()
    is_rush_hour = (pl.col("tpep_pickup_datetime").dt.weekday() <= 5) & (
        pickup_time.is_between(pl.time(6, 30), pl.time(9, 30))
        | pickup_time.is_between(pl.time(15, 30), pl.time(20))
    )
    return (
        rides.join(
            zones.rename({"LocationID": "PULocationID", "borough": "pickup_borough"}),
            on="PULocationID",
            how="left",
        )
        .join(
            zones.rename({"LocationID": "DOLocationID", "borough": "dropoff_borough"}),
            on="DOLocationID",
            how="left",
        )
        .drop("PULocationID", "DOLocationID")
        .with_columns(
            pl.col("payment_type").replace_strict(
                {1: "card", 2: "cash"}, default="other", return_dtype=PAYMENT_TYPE
            ),
            (pl.col("airport_fee") > 0).alias("is_airport_ride"),
            is_rush_hour.alias("is_rush_hour"),
        )
    )

In [8]:
read_rides().pipe(clean).pipe(transform, read_zones()).head().collect()

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,pickup_borough,dropoff_borough,is_airport_ride,is_rush_hour
u8,datetime[μs],datetime[μs],u8,f64,u8,enum,enum,f64,f64,f64,f64,f64,f64,f64,f64,f64,enum,enum,bool,bool
2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,"""N""","""cash""",17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0,"""manhattan""","""manhattan""",false,false
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,"""N""","""card""",10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0,"""manhattan""","""manhattan""",false,false
1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,"""N""","""card""",23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0,"""manhattan""","""manhattan""",false,false
1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,"""N""","""card""",10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0,"""manhattan""","""manhattan""",false,false
1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,"""N""","""card""",7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0,"""manhattan""","""manhattan""",false,false


## 4. Feature extraction

In [9]:
def dummies(col: str, dtype: pl.Enum) -> list[pl.Expr]:
    return [
        (pl.col(col) == value).alias(f"{col}_{value}") for value in dtype.categories
    ]


def extract_features(rides: pl.LazyFrame) -> pl.LazyFrame:
    return (
        rides.with_columns(
            pl.col("tpep_pickup_datetime").dt.date().alias("date"),
            *dummies("payment_type", PAYMENT_TYPE),
            *dummies("pickup_borough", BOROUGH),
            *dummies("dropoff_borough", BOROUGH),
        )
        .group_by("date")
        .agg(
            pl.len().alias("n_rides"),
            pl.col("is_airport_ride").sum().alias("n_airport_rides"),
            pl.col("is_rush_hour").sum().alias("n_rush_hour_rides"),
            pl.col("^(payment_type|pickup_borough|dropoff_borough)_.*$")
            .sum()
            .name.prefix("n_"),
            pl.col("fare_amount").mean().alias("avg_fare_amount"),
            pl.col("trip_distance").median().alias("median_trip_distance"),
            pl.col("total_amount").sum().alias("total_amount_sum"),
            *[
                pl.col("total_amount")
                .filter(pl.col("payment_type") == payment)
                .sum()
                .alias(f"total_amount_{payment}")
                for payment in PAYMENT_TYPE.categories
            ],
            pl.col("congestion_surcharge").sum().alias("congestion_surcharge_sum"),
            pl.col("passenger_count")
            .sum()
            .cast(pl.UInt32)
            .alias("passenger_count_sum"),
        )
        .with_columns(
            pl.col("date").dt.quarter().alias("quarter"),
            pl.col("date").dt.month().alias("month"),
            pl.col("date").dt.day().alias("day_of_month"),
            pl.col("date").dt.weekday().alias("day_of_week"),
            (pl.col("date").dt.weekday() >= 6).alias("is_weekend"),
        )
        .sort("date")
    )

In [10]:
dataset = (
    read_rides()
    .pipe(clean)
    .pipe(transform, read_zones())
    .pipe(extract_features)
    .collect()
)
dataset.write_parquet("dataset.parquet")